# Sol en casa

Quiero saber exactamente cuanta luz entra a un inmueble en cada momento determinado del año. Es viable hacer esto si te digo las coordenadas del polígono del inmueble, latitud longitud y elevación (es un segundo piso) así como coordenadas de los ventanales, y las coordenadas de polígonos vecinos que podrían obstruirla? Quiero hacer una aplicación en poorly dash donde pueda usar un control deslizante de horas del día y fechas para saber la posición exacta del sol

In [1]:
import folium
import pandas as pd

# Parse coordinates
coords = [
    [19.39178725955838, -99.1618464534615],
    [19.391938799909664, -99.16180872404323],
    [19.39192617155245, -99.16175030429882],
    [19.39176889102137, -99.16177829709301]
]

# Create map centered on polygon
center = [sum(c[0] for c in coords) / len(coords), sum(c[1] for c in coords) / len(coords)]
m = folium.Map(location=center, zoom_start=18)

# Add polygon to map
folium.Polygon(coords, color='red', fill=True, fillColor='red', fillOpacity=0.3).add_to(m)

# Display map
m

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import pytz
from pysolar.solar import get_altitude, get_azimuth
import plotly.graph_objects as go
import plotly.express as px
from dash import Dash, dcc, html, Input, Output, State
import threading

# Coordenadas del inmueble (en Ciudad de México)
building_coords = [
    [19.39178725955838, -99.1618464534615],
    [19.391938799909664, -99.16180872404323],
    [19.39192617155245, -99.16175030429882],
    [19.39176889102137, -99.16177829709301]
]

# Altura del volumen en metros
building_height = 3.0

# Latitud y longitud promedio del edificio
latitude = sum(c[0] for c in building_coords) / len(building_coords)
longitude = sum(c[1] for c in building_coords) / len(building_coords)

print(f"Coordenadas promedio del edificio: {latitude}, {longitude}")
print(f"Altura del volumen: {building_height}m")

# Crear zona horaria para CDMX
tz = pytz.timezone('America/Mexico_City')

def get_sun_position(date, hour, latitude, longitude):
    """
    Calcula la posición del sol (azimut y elevación) para una fecha y hora determinadas
    """
    dt = tz.localize(datetime(date.year, date.month, date.day, int(hour), int((hour % 1) * 60)))
    
    altitude = get_altitude(latitude, longitude, dt)  # En grados
    azimuth = get_azimuth(latitude, longitude, dt)    # En grados
    
    return altitude, azimuth

def normalize_coords(coords):
    """
    Normaliza las coordenadas geográficas (lat, lon) a un sistema 2D local
    Usamos una proyección simple para visualización
    """
    # Convertir a metros usando aproximación
    # 1 grado de latitud ≈ 111 km
    # 1 grado de longitud ≈ 111 km * cos(latitud)
    
    lat_meters_per_degree = 111000
    lon_meters_per_degree = 111000 * np.cos(np.radians(latitude))
    
    ref_lat, ref_lon = latitude, longitude
    
    local_coords = []
    for lat, lon in coords:
        x = (lon - ref_lon) * lon_meters_per_degree
        y = (lat - ref_lat) * lat_meters_per_degree
        local_coords.append([x, y])
    
    return np.array(local_coords)

def calculate_shadow(coords_2d, altitude, azimuth, height):
    """
    Calcula la proyección de sombra del volumen 3D sobre el plano de piso
    """
    if altitude <= 0:
        # Sol bajo el horizonte, sombra máxima
        return None
    
    # Convertir azimut y altitud a vector de luz
    alt_rad = np.radians(altitude)
    az_rad = np.radians(azimuth)
    
    # Vector de dirección del sol (normalizado)
    sun_x = np.sin(az_rad) * np.cos(alt_rad)
    sun_y = np.cos(az_rad) * np.cos(alt_rad)
    sun_z = np.sin(alt_rad)
    
    # Puntos de las esquinas de la base
    shadow_points = []
    
    for point_2d in coords_2d:
        x, y = point_2d
        
        # Punto del techo (altura = 3m)
        roof_x = x
        roof_y = y
        roof_z = height
        
        # Proyectar hacia el suelo siguiendo la dirección del sol
        # Encontrar donde la línea desde el punto del techo intersecta z=0
        # Línea: P = (roof_x, roof_y, roof_z) + t * (sun_x, sun_y, sun_z)
        # Cuando z = 0: roof_z + t * sun_z = 0 => t = -roof_z / sun_z
        
        if sun_z != 0:
            t = -roof_z / sun_z
            shadow_x = roof_x + t * sun_x
            shadow_y = roof_y + t * sun_y
            shadow_points.append([shadow_x, shadow_y])
    
    return np.array(shadow_points)

# Crear figura de prueba
print("Funciones creadas exitosamente")

Coordenadas promedio del edificio: 19.391855280510466, -99.16179594472413
Altura del volumen: 3.0m
Funciones creadas exitosamente


In [2]:
def create_3d_visualization(date, hour):
    """
    Crea una visualización 3D que muestra:
    - El volumen del edificio
    - La sombra proyectada
    - La posición del sol
    """
    
    # Obtener posición del sol
    altitude, azimuth = get_sun_position(date, hour, latitude, longitude)
    
    # Normalizar coordenadas
    coords_2d = normalize_coords(building_coords)
    
    # Crear figura
    fig = go.Figure()
    
    # 1. Dibujar la base del volumen (ras de piso)
    base_x = np.append(coords_2d[:, 0], coords_2d[0, 0])
    base_y = np.append(coords_2d[:, 1], coords_2d[0, 1])
    base_z = np.zeros_like(base_x)
    
    fig.add_trace(go.Scatter3d(
        x=base_x, y=base_y, z=base_z,
        mode='lines',
        name='Base del volumen',
        line=dict(color='blue', width=3)
    ))
    
    # 2. Dibujar el techo del volumen
    roof_x = np.append(coords_2d[:, 0], coords_2d[0, 0])
    roof_y = np.append(coords_2d[:, 1], coords_2d[0, 1])
    roof_z = np.ones_like(roof_x) * building_height
    
    fig.add_trace(go.Scatter3d(
        x=roof_x, y=roof_y, z=roof_z,
        mode='lines',
        name='Techo del volumen',
        line=dict(color='darkblue', width=3)
    ))
    
    # 3. Dibujar las paredes verticales
    for i in range(len(coords_2d)):
        next_i = (i + 1) % len(coords_2d)
        fig.add_trace(go.Scatter3d(
            x=[coords_2d[i, 0], coords_2d[next_i, 0]],
            y=[coords_2d[i, 1], coords_2d[next_i, 1]],
            z=[0, 0],
            mode='lines',
            showlegend=False,
            line=dict(color='blue', width=2)
        ))
        fig.add_trace(go.Scatter3d(
            x=[coords_2d[i, 0], coords_2d[i, 0]],
            y=[coords_2d[i, 1], coords_2d[i, 1]],
            z=[0, building_height],
            mode='lines',
            showlegend=False,
            line=dict(color='blue', width=2)
        ))
    
    # 4. Dibujar la sombra si el sol está sobre el horizonte
    if altitude > 0:
        shadow_coords = calculate_shadow(coords_2d, altitude, azimuth, building_height)
        
        if shadow_coords is not None:
            # Dibujar puntos de la sombra con marcadores semitransparentes
            fig.add_trace(go.Scatter3d(
                x=shadow_coords[:, 0], 
                y=shadow_coords[:, 1], 
                z=np.zeros(len(shadow_coords)),
                mode='markers',
                marker=dict(
                    size=8,
                    color='red',
                    opacity=0.6
                ),
                name='Proyección de vértices',
                showlegend=False
            ))
            
            # Dibujar contorno de la sombra cerrado
            shadow_x_closed = np.append(shadow_coords[:, 0], shadow_coords[0, 0])
            shadow_y_closed = np.append(shadow_coords[:, 1], shadow_coords[0, 1])
            shadow_z_closed = np.zeros_like(shadow_x_closed)
            
            fig.add_trace(go.Scatter3d(
                x=shadow_x_closed, y=shadow_y_closed, z=shadow_z_closed,
                mode='lines',
                name='Borde de sombra',
                line=dict(color='darkred', width=3)
            ))
    
    # Configurar el layout
    fig.update_layout(
        title=f"Simulación Solar - {date.strftime('%Y-%m-%d')} - {hour:.1f}h<br>Altitud: {altitude:.1f}° | Azimut: {azimuth:.1f}°",
        scene=dict(
            xaxis=dict(title='Longitud (m)', backgroundcolor="rgb(230, 230,230)"),
            yaxis=dict(title='Latitud (m)', backgroundcolor="rgb(230, 230,230)"),
            zaxis=dict(title='Altura (m)', backgroundcolor="rgb(230, 230,230)"),
            aspectmode='data'
        ),
        width=1000,
        height=700,
        showlegend=True
    )
    
    return fig

# Prueba con una fecha y hora específica
test_date = datetime(2026, 1, 17)  # Hoy
test_hour = 12  # Mediodía

fig_test = create_3d_visualization(test_date, test_hour)
fig_test.show()

In [4]:
from jupyter_dash import JupyterDash

# Crear la aplicación Dash
app = JupyterDash(__name__)

app.layout = html.Div([
    html.H1("Simulador Solar - Sol en Casa", style={'textAlign': 'center', 'marginBottom': 30}),
    
    html.Div([
        # Control de Fecha
        html.Div([
            html.Label("Selecciona la fecha:", style={'fontWeight': 'bold'}),
            dcc.DatePickerSingle(
                id='date-picker',
                date=datetime(2026, 1, 17),
                display_format='YYYY-MM-DD',
                style={'width': '100%', 'padding': '10px'}
            ),
        ], style={'width': '30%', 'display': 'inline-block', 'marginRight': '5%', 'verticalAlign': 'top'}),
        
        # Control de Hora con Slider
        html.Div([
            html.Label("Hora del día (0-24h):", style={'fontWeight': 'bold'}),
            dcc.Slider(
                id='hour-slider',
                min=0,
                max=24,
                step=0.5,
                value=12,
                marks={i: f'{i}h' for i in range(0, 25, 2)},
                tooltip={"placement": "bottom", "always_visible": True}
            ),
            html.Div(id='hour-display', style={'marginTop': '10px', 'fontSize': '14px', 'color': '#666'})
        ], style={'width': '65%', 'display': 'inline-block', 'verticalAlign': 'top'}),
    ], style={'marginBottom': '30px', 'padding': '20px', 'backgroundColor': '#f5f5f5', 'borderRadius': '5px'}),
    
    # Información del sol
    html.Div(id='sun-info', style={'marginBottom': '20px', 'padding': '15px', 'backgroundColor': '#fff3cd', 'borderRadius': '5px'}),
    
    # Gráfico 3D
    html.Div([
        dcc.Graph(id='3d-visualization')
    ], style={'marginBottom': '30px'}),
    
    # Información de la sombra
    html.Div(id='shadow-info', style={'padding': '15px', 'backgroundColor': '#e7f3ff', 'borderRadius': '5px'})
    
], style={'padding': '20px', 'fontFamily': 'Arial, sans-serif', 'maxWidth': '1200px', 'margin': '0 auto'})

@app.callback(
    [Output('3d-visualization', 'figure'),
     Output('hour-display', 'children'),
     Output('sun-info', 'children'),
     Output('shadow-info', 'children')],
    [Input('date-picker', 'date'),
     Input('hour-slider', 'value')]
)
def update_visualization(date_str, hour):
    # Convertir string de fecha a datetime
    date = pd.to_datetime(date_str).date()
    date = datetime(date.year, date.month, date.day)
    
    # Crear figura
    fig = create_3d_visualization(date, hour)
    
    # Obtener información del sol
    altitude, azimuth = get_sun_position(date, hour, latitude, longitude)
    
    # Información de la hora
    hour_display = f"Hora: {hour:.1f}h ({int(hour)}:{int((hour % 1) * 60):02d})"
    
    # Información del sol
    if altitude < 0:
        sun_info = html.Div([
            html.P("☀️ El sol está bajo el horizonte - Noche completa"),
            html.P(f"Altitud: {altitude:.1f}°")
        ])
        shadow_info = html.Div([
            html.P("Sin iluminación directa del sol")
        ])
    elif altitude < 5:
        sun_info = html.Div([
            html.P("🌅 Luz rasante - Amanecer o atardecer"),
            html.P(f"Altitud: {altitude:.1f}° | Azimut: {azimuth:.1f}°")
        ])
        shadow_info = html.Div([
            html.P("Sombra muy pronunciada y extendida")
        ])
    else:
        sun_info = html.Div([
            html.P(f"☀️ Sol a {altitude:.1f}° sobre el horizonte"),
            html.P(f"Azimut: {azimuth:.1f}° (0°=N, 90°=E, 180°=S, 270°=O)")
        ])
        
        # Calcular área de sombra
        shadow_coords = calculate_shadow(normalize_coords(building_coords), altitude, azimuth, building_height)
        if shadow_coords is not None:
            # Calcular área usando el método del polígono
            def polygon_area(points):
                x = points[:, 0]
                y = points[:, 1]
                return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
            
            shadow_area = polygon_area(shadow_coords)
            shadow_info = html.Div([
                html.P(f"Área de sombra: {shadow_area:.2f} m²")
            ])
        else:
            shadow_info = html.Div([html.P("No hay sombra calculada")])
    
    return fig, hour_display, sun_info, shadow_info

# Ejecutar la aplicación en Jupyter
app.run(mode='inline', port=8050)